# VXN/VIX 单一信息增量

沿用固定 H5 预测与配置方法，公平比较25价格特征与26特征；只增加初次发布 log(VXN/VIX)。先完成原生来源和样本资格。参数与判定见 plan.json。

In [1]:
from pathlib import Path
import os,sys,json
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
ROOT=Path.cwd()
if not (ROOT/'src').is_dir(): ROOT=ROOT.parents[1]
sys.path.insert(0,str(ROOT))
CARD=ROOT/'research/vxn_vix_information_increment_20261011'
import qlib
qlib.init(provider_uri=str(ROOT/'data/qlib/published_futu_affine_20260925_v1'),region='us')
from src.core.recorder_sources import load_verified_recorder_objects
from src.workflow.vxn_vix_information_increment import prepare_information_increment_card
if not (CARD/'summary.json').exists():
    summary=prepare_information_increment_card(root=ROOT,card=CARD)
else:
    summary=json.loads((CARD/'summary.json').read_text())
    assert summary['plan_sha256']==__import__('hashlib').sha256((CARD/'plan.json').read_bytes()).hexdigest()
    if summary['status']=='prefit_qualified_not_promoted':
        objects,audit=load_verified_recorder_objects(CARD,expected_recorder_id=summary['recorder_id'],artifacts=['prefit/qualification.pkl'])
        assert objects['prefit/qualification.pkl']['all_arm_identities_equal']
print('Native qualified:',summary['status'])


[73843:MainThread](2026-10-11 16:51:07,444) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/11 16:51:07 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[73843:MainThread](2026-10-11 16:51:07,901) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[73843:MainThread](2026-10-11 16:51:07,901) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Native qualified: complete_not_promoted


In [2]:
from src.workflow.vxn_vix_information_increment import run_information_increment
attempt=CARD/'execution_attempt.json'
if attempt.exists():
    state=json.loads(attempt.read_text())
    assert state['status']=='completed', 'blocked: preserve existing attempt; do not refit'
    summary=json.loads((CARD/'summary.json').read_text())
    objects,audit=load_verified_recorder_objects(CARD,expected_recorder_id=summary['recorder_id'],artifacts=['aggregate/final_summary.pkl'])
    assert objects['aggregate/final_summary.pkl']==summary
else:
    summary=run_information_increment(root=ROOT,card=CARD,execution_plan_path=CARD/'execution_plan.json')
print(json.dumps(summary,ensure_ascii=False,indent=2))
from IPython.display import Markdown,display
display(Markdown((CARD/'conclusion.md').read_text()))


{
  "experiment": "vxn_vix_information_increment_20261011",
  "recorder_id": "79303313446a4c8081f4b0b773c348d6",
  "recorder_uri": "/Users/rziiiii/dev/workspace/research/vxn_vix_information_increment_20261011/execution_recorder",
  "recorder_local_dir": "/Users/rziiiii/dev/workspace/research/vxn_vix_information_increment_20261011/execution_recorder/451822175291100435/79303313446a4c8081f4b0b773c348d6",
  "status": "complete_not_promoted",
  "promotion_decision": "not_promoted",
  "promotion_eligible": false,
  "scientific_promotion": false,
  "fit_calls": 100,
  "native_cases": 12,
  "historical_gate_pass": false,
  "diagnostic_gate_pass": true,
  "incremental_portfolio_pass": false,
  "incremental_information_pass": false,
  "route_decision": "end_vxn_vix_route_no_further_search",
  "source": {
    "provider_uri": "data/qlib/published_futu_affine_20260925_v1",
    "execution_provider_uri": "data/qlib/legacy_soxl_execution_20260925_v1",
    "information_provider_uri": "data/qlib/alfred_

# VXN/VIX 单一信息增量：结果

固定100次拟合、12个原生账户场景全部完成；控制组25特征与候选26特征使用相同来源资格、严格成熟样本。
2016两组按H5节奏退回SOXX；未知行不预测、不删日历，非调仓日精确HOLD。

- development/10bp：净收益 315.35%，相对SOXX终值 0.975576，最大回撤 -47.37%。
- reused_diagnostic/10bp：净收益 426.61%，相对SOXX终值 1.057195，最大回撤 -42.79%。
- development/25bp：净收益 325.11%，相对SOXX终值 1.000000，最大回撤 -47.37%。
- reused_diagnostic/25bp：净收益 452.81%，相对SOXX终值 1.111439，最大回撤 -42.79%。

历史门 False；复用诊断门 True；相对匹配控制组组合增量 False；固定超额响应MSE增量 False。
不晋升；路线决定：end_vxn_vix_route_no_further_search。
费用情景会改变优化动作：诊断区间10bp于2026-05-27转现金，25bp于2026-06-03转现金，均于06-10买回，5个持仓日不同。较高费用下的更高终值不是同路径成本改善。候选全程SOXL权重为0，收益差异来自SOXX/现金时机；没有验证SOXL配置价值。
逐窗口SOXL、SOXX及超额响应MSE/IC见 forecast_information.csv；IC仅为描述，未做显著性认证。历史复用、日期级初次发布、固定HAC5均不构成独立前向证据。
失败即停止此来源路线，不改变期限、alpha、阈值、特征符号或风险门。若全部历史门通过，仍需新冻结身份和至少60日独立前向证据，不能据此立即上线。
Recorder：79303313446a4c8081f4b0b773c348d6；FINISHED。


# VXN/VIX 单一信息增量：结果

固定100次拟合、12个原生账户场景全部完成；控制组25特征与候选26特征使用相同来源资格、严格成熟样本。
2016两组按H5节奏退回SOXX；未知行不预测、不删日历，非调仓日精确HOLD。

- development/10bp：净收益 315.35%，相对SOXX终值 0.975576，最大回撤 -47.37%。
- reused_diagnostic/10bp：净收益 426.61%，相对SOXX终值 1.057195，最大回撤 -42.79%。
- development/25bp：净收益 325.11%，相对SOXX终值 1.000000，最大回撤 -47.37%。
- reused_diagnostic/25bp：净收益 452.81%，相对SOXX终值 1.111439，最大回撤 -42.79%。

历史门 False；复用诊断门 True；相对匹配控制组组合增量 False；固定超额响应MSE增量 False。
不晋升；路线决定：end_vxn_vix_route_no_further_search。
费用情景会改变优化动作：诊断区间10bp于2026-05-27转现金，25bp于2026-06-03转现金，均于06-10买回，5个持仓日不同。较高费用下的更高终值不是同路径成本改善。候选全程SOXL权重为0，收益差异来自SOXX/现金时机；没有验证SOXL配置价值。
逐窗口SOXL、SOXX及超额响应MSE/IC见 forecast_information.csv；IC仅为描述，未做显著性认证。历史复用、日期级初次发布、固定HAC5均不构成独立前向证据。
失败即停止此来源路线，不改变期限、alpha、阈值、特征符号或风险门。若全部历史门通过，仍需新冻结身份和至少60日独立前向证据，不能据此立即上线。
Recorder：79303313446a4c8081f4b0b773c348d6；FINISHED。
